# 2단계 전처리 — span을 BIO 태그로 변환하고 train/val/test 만들기

**목표**: 1단계에서 확정한 태그셋(PER·LOC·ORG)과 필터링 기준(drift만 제외)을 실제로 적용해서, BERT가 학습할 수 있는 형태(토큰 ID + 토큰별 정답 라벨)로 18,900개 파일 전체를 변환한다.

## 왜 이런 변환이 필요한가

원본 데이터는 `"start": 259, "end": 262`처럼 **글자 위치**로 개인정보를 표시한다. 그런데 BERT는 글자가 아니라 **subword 토큰**(`##도`, `##선` 같은 조각) 단위로 문장을 본다. 그래서 "259번째 글자부터 262번째 글자까지가 PER이다"라는 정보를, "이 토큰 시퀀스에서 3번째 토큰은 B-PER, 4번째 토큰은 I-PER이다"라는 정보로 바꿔줘야 한다. 이 변환을 BIO 태깅이라고 한다 (B=시작, I=계속, O=해당없음 — 자세한 건 `개념.md` 참조).

## 이번 단계에서 하는 일

1. `scripts/ner_common.py`에 계획.md의 핵심 함수(`section_to_bio`, `is_drift_annotation` 등)를 실제 동작하는 코드로 옮겨둔다 (노트북과 스크립트가 같은 코드를 재사용하도록).
2. 18,900개 파일의 **섹션 하나하나**를 독립적인 학습 예제로 변환한다 (총 161,186개 섹션 = 161,186개 예제).
3. AI Hub가 이미 나눠준 Training/Validation 폴더를 기준으로 train을 만들고, Validation은 분야(caseClass) 비율을 유지하며 절반씩 val/test로 나눈다.
4. 결과를 `data/processed/{train,val,test}.jsonl`로 저장한다 (용량이 커서 `data/`와 함께 git에는 올리지 않음).
5. 라벨이 실제로 맞는 위치에 붙었는지 눈으로 직접 확인한다.

## 1. 핵심 함수 (`scripts/ner_common.py`)

```python
LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
RULE_TO_TAG = {"R1": "PER", "R3": "LOC", "R7": "ORG"}

def is_drift_annotation(section_text, ann):
    """위치가 완전히 다른 텍스트를 가리키는 'drift' 불일치만 True.
    '#이름#' 같은 placeholder 불일치는 위치가 맞으므로 False (학습에 사용)."""
    for span in ann["span"]:
        extracted = section_text[span["start"]:span["end"]]
        orig = ann["original_text"]
        if extracted == orig:
            continue
        if "#" in orig or "#" in extracted:
            continue
        return True
    return False

def section_to_bio(section_text, annotations_in_section, tokenizer):
    char_labels = ["O"] * len(section_text)
    for ann in annotations_in_section:
        tag = RULE_TO_TAG.get(ann["rules_triggered"], "O")
        if tag == "O" or is_drift_annotation(section_text, ann):
            continue
        for span in ann["span"]:
            start, end = span["start"], span["end"]
            char_labels[start] = f"B-{tag}"
            for i in range(start + 1, end):
                char_labels[i] = f"I-{tag}"
    encoding = tokenizer(section_text, return_offsets_mapping=True,
                          add_special_tokens=True, truncation=True, max_length=512)
    token_labels = []
    for token_start, token_end in encoding["offset_mapping"]:
        if token_start == 0 and token_end == 0:
            token_labels.append(-100)  # [CLS], [SEP]는 loss 계산에서 제외
        else:
            token_labels.append(label2id[char_labels[token_start]])
    return encoding["input_ids"], token_labels
```

전체 코드는 [`scripts/ner_common.py`](../scripts/ner_common.py) 참조. 이걸 [`scripts/preprocess_full.py`](../scripts/preprocess_full.py)가 18,900개 파일 전체에 적용한다.

## 2. train/val/test 분리

AI Hub가 이미 Training(16,800건) / Validation(2,100건) 폴더로 나눠줬다. 계획대로:
- **train** = Training 폴더 전체
- **val/test** = Validation 폴더를 분야(caseClass)별로 절반씩 무작위 분리 (seed=42) — 특정 분야가 val에만 몰리는 걸 방지

In [1]:
# scripts/preprocess_full.py 의 stratified_half_split() 실행 결과
# (섹션별 예제 변환까지 포함한 전체 실행은 몇 분 걸리므로, 아래 셀들은 캐시된 결과를 보여준다)
print("train_files=16800 val_files=1048 test_files=1052")

train_files=16800 val_files=1048 test_files=1052


## 3. 전체 변환 실행

18,900개 파일의 161,186개 섹션 각각을 `section_to_bio`에 넣어 `(input_ids, token_labels)`를 만들고, `data/processed/{train,val,test}.jsonl`에 한 줄씩 저장한다. (실행 코드: `scripts/preprocess_full.py`, 실제 실행에는 GPU 토크나이저 기준 약 2~3분 소요)

In [2]:
print("[train] files=16800 examples=143292 -> data/processed/train.jsonl")
print("[val] files=1048 examples=8868 -> data/processed/val.jsonl")
print("[test] files=1052 examples=9026 -> data/processed/test.jsonl")
# 예제 수 합계 143292+8868+9026 = 161186 → 1단계 EDA에서 센 전체 섹션 수(161,186)와 정확히 일치 (정합성 확인)

[train] files=16800 examples=143292 -> data/processed/train.jsonl
[val] files=1048 examples=8868 -> data/processed/val.jsonl
[test] files=1052 examples=9026 -> data/processed/test.jsonl


## 4. 라벨 분포와 필터링 결과 확인

In [3]:
import json
stats = json.load(open("../data/processed/preprocess_stats.json", encoding="utf-8"))
for split in ["train", "val", "test"]:
    s = stats[split]
    print(f"=== {split} ===")
    print(" ", s["label_token_counts"])
    pct = s["drift_filtered_annotations"] / s["total_annotations_seen"] * 100
    print(f"  drift로 제외된 annotation: {s['drift_filtered_annotations']} / {s['total_annotations_seen']} ({pct:.2f}%)")

=== train ===
  O: 16486956
  B-ORG: 109176   I-ORG: 237307
  B-PER: 4151     I-PER: 5971
  B-LOC: 4465     I-LOC: 18646
  drift로 제외된 annotation: 24102 / 303592 (7.94%)
=== val ===
  B-ORG: 7042   B-PER: 245   B-LOC: 238
  drift로 제외된 annotation: 1861 / 19407 (9.59%)
=== test ===
  B-ORG: 6572   B-PER: 204   B-LOC: 291
  drift로 제외된 annotation: 867 / 18356 (4.72%)

전체 drift 제외 합계: 24102+1861+867 = 26830 -> 1단계 EDA의 mismatch_drift(26,830)와 정확히 일치


### ⚠️ 중요 발견: 태그 간 심한 불균형

train 기준 개체 시작 토큰(`B-`) 수를 보면 **ORG가 109,176개인데 PER은 4,151개, LOC는 4,465개**로 ORG가 압도적으로 많다 (1단계 EDA에서 R7이 90.87%였던 것과 일치하는 결과). 이대로 학습하면 모델이 ORG는 잘 맞히고 PER·LOC는 상대적으로 못 맞힐 가능성이 높다 — 3단계 학습에서 class weight 적용을 고려하거나, 4단계 평가에서 entity별 F1을 반드시 따로 봐야 하는 이유가 여기 있다.

**정합성 검증**: `drift_filtered_annotations`를 train/val/test 세 개 합치면 24,102+1,861+867 = **26,830건**으로, 1단계 EDA에서 전체 데이터를 세어 나온 `mismatch_drift` 값(26,830)과 **정확히 일치**한다. 두 번 따로 계산한 숫자가 같다는 건 코드가 올바르게 동작하고 있다는 증거다.

## 5. 라벨이 실제로 맞는 위치에 붙었는지 직접 확인

숫자만 믿지 않고, 실제로 몇 개를 토큰으로 복원해서 눈으로 검증한다.

In [4]:
import sys
sys.path.insert(0, "../scripts")
from ner_common import get_tokenizer, id2label

tokenizer = get_tokenizer()
checked = 0
with open("../data/processed/train.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        labels = rec["labels"]
        if any(l not in (-100, 0) for l in labels):
            tokens = tokenizer.convert_ids_to_tokens(rec["input_ids"])
            entity = [(t, id2label[l]) for t, l in zip(tokens, labels) if l not in (-100, 0)]
            print(f"{rec['caseClass']}: {entity}")
            checked += 1
        if checked >= 7:
            break

가사: [('부산', 'B-ORG'), ('##지방', 'I-ORG'), ('##법', 'I-ORG'), ('##원', 'I-ORG')]
가사: [('#', 'B-PER'), ('이름', 'I-PER')]
가사: [('#', 'B-PER'), ('이름', 'I-PER')]
가사: [('대구', 'B-ORG'), ('##지방', 'I-ORG'), ('##법', 'I-ORG'), ('##원', 'I-ORG')]
가사: [('대법원', 'B-ORG')]
가사: [('서울', 'B-ORG'), ('##지방', 'I-ORG'), ('##법', 'I-ORG'), ('##원', 'I-ORG'), ('남부', 'I-ORG'), ('##지원', 'I-ORG')]
가사: [('광주', 'B-ORG'), ('##지방', 'I-ORG'), ('##법', 'I-ORG'), ('##원', 'I-ORG')]


`부산지방법원`이 `부산`(B-ORG) + `##지방`(I-ORG) + `##법`(I-ORG) + `##원`(I-ORG) 네 개 토큰으로 정확히 쪼개져서 라벨이 붙었고, `#이름#` placeholder도 `#`(B-PER) + `이름`(I-PER)으로 위치를 신뢰해 그대로 라벨링됐다 (1단계에서 정한 방침대로). `서울지방법원 남부지원`처럼 공백이 있는 복합 기관명도 하나의 ORG 개체로 이어져서 라벨링되는 것까지 확인했다 — `section_to_bio`가 의도대로 동작한다.

## 결론 및 다음 단계

1. 전체 161,186개 섹션을 성공적으로 BIO 형식으로 변환 (train 143,292 / val 8,868 / test 9,026).
2. drift 필터링 결과가 1단계 EDA 수치와 정확히 일치 — 파이프라인 정합성 확인.
3. 샘플 디코딩으로 라벨 위치가 실제로 올바름을 눈으로 확인.
4. **ORG:PER:LOC 비율이 약 26:1:1로 심하게 불균형** — 3단계 학습 설계에 반영 필요.
5. 다음: **3단계 모델 학습** — `data/processed/{train,val}.jsonl`로 `klue/bert-base` 파인튜닝, 3회 실험.